# Group specparam tables

Collects the specparam results of all subjects of one run and flags subjects for
exclusion. Runs once per run, after steps 01-05: `run_pipeline.py` executes it
at the end of a full run (`make group` runs it on its own), or open it and run it
here. The run folder comes from `settings.toml`; the tables are written to
`<outputs_root>/group/`:

- `all_subjects_specparam.csv`: every row of every subject (one row per peak)
- `unique_specparam_results.csv`: one row per channel (offset, exponent, fit)

Both channel tables have an `interpolated` column: True for channels that were
marked bad and interpolated in step 04, so their values come from neighbouring
channels.
- `participant_exclusion_report.csv`: per subject, the specparam fit quality
  (`specparam_n_channels`, `specparam_good_fit_share`, `specparam_status`,
  criteria in `[group]`), the fit error beside it (`specparam_error_median`,
  `specparam_error_z`; shown only, not part of the rule), then the
  preprocessing counts from step 04

In [ ]:
# set by run_pipeline.py (papermill); None means: find settings.toml from cwd
config_path = None
# set by run_pipeline.py, e.g. a test folder for derivatives_root / outputs_root
path_overrides = None

In [ ]:
import json

import pandas as pd
from IPython.display import display

from spectral.runner import PipelineConfig
from spectral.utils import ProjectPaths, load_config

cfg = load_config(config_path)
# Paths and subject list only; the group step runs no subject steps itself.
run = PipelineConfig.from_settings(config_path=config_path, steps=[], **(path_overrides or {}))
subjects = run.subjects()
paths = {s: ProjectPaths(s, config_path=config_path, path_overrides=path_overrides)
         for s in subjects}
out_dir = paths[subjects[0]].outputs / "group"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"{len(subjects)} subjects in {run.bids_root}")
print(f"writing to {out_dir}")

## Collect

In [ ]:
# One file per subject from step 05. The subject is taken from the folder, not
# parsed from the file name, and only *-specparam.csv files are read.
tables, missing = [], []
for p in paths.values():
    f = p.specparam / f"{p.subject}-specparam.csv"
    if f.exists():
        tables.append(pd.read_csv(f).assign(subject_id=p.subject))
    else:
        missing.append(p.subject)
all_rows = pd.concat(tables, ignore_index=True)
all_rows = all_rows[["subject_id"] + [c for c in all_rows.columns if c != "subject_id"]]

# Step 04 record per subject: preprocessing counts and the channels that were
# interpolated (marked bad in step 01 or 02, then replaced in all epochs by
# interpolation from neighbouring channels). autoreject's per-epoch repairs
# are not part of this list.
meta = [pd.read_csv(f, dtype={"subject_id": str, "bad_channels": str})
        for p in paths.values()
        if (f := p.specparam / f"{p.subject}_ica_metadata.csv").exists()]
meta = pd.concat(meta, ignore_index=True).drop(columns=["timestamp"], errors="ignore")
meta["subject_id"] = "sub-" + meta["subject_id"].str.zfill(3)
interpolated = {s: set(b.split(",")) if isinstance(b, str) else set()
                for s, b in zip(meta["subject_id"], meta["bad_channels"])}
no_record = sorted(set(all_rows["subject_id"]) - set(interpolated))
all_rows.insert(all_rows.columns.get_loc("ch") + 1, "interpolated",
                [ch in interpolated.get(s, set())
                 for s, ch in zip(all_rows["subject_id"], all_rows["ch"])])

# Steps that failed in this run, from the runner's status file
status = json.loads(run.status_file.read_text()) if run.status_file.exists() else {}
failed = {f"sub-{s}": [k for k, v in steps.items()
                       if isinstance(v, dict) and v.get("status") == "failed"]
          for s, steps in status.items()}
failed = {s: k for s, k in failed.items() if k}

print(f"{len(tables)} subjects with results")
print(f"no results: {missing or 'none'}")
print(f"failed steps: {failed or 'none'}")
print(f"results without a step 04 record: {no_record or 'none'}")

## One row per channel

In [ ]:
# The specparam file has one row per peak, with the aperiodic values repeated on
# each row of a channel.
per_channel = (all_rows.drop(columns=["CF", "PW", "BW"], errors="ignore")
               .drop_duplicates(subset=["subject_id", "ch"])
               .reset_index(drop=True))
print(f"{len(all_rows)} rows (peaks), {len(per_channel)} channels")

# The flagged channels per subject must equal the step 04 count.
n_flagged = per_channel.groupby("subject_id")["interpolated"].sum()
n_step04 = meta.set_index("subject_id")["nr_interpolated_channels"].reindex(n_flagged.index)
mismatch = n_flagged[n_flagged != n_step04].index.tolist()
print(f"interpolated channels: {int(n_flagged.sum())} in {int((n_flagged > 0).sum())} subjects; "
      f"differs from step 04 count: {mismatch or 'none'}")

## Exclusion report

In [ ]:
# specparam_* columns describe the spectral fits (step 05): the number of
# channels fitted, the share of them with R^2 >= min_r_squared, and the
# resulting Keep/Exclude. The step 04 preprocessing counts follow after them.
_g = cfg["group"]
report = (per_channel.assign(good_fit=per_channel["gof_rsquared"] >= _g["min_r_squared"])
          .groupby("subject_id")
          .agg(specparam_n_channels=("ch", "size"),
               specparam_good_fit_share=("good_fit", "mean"))
          .reset_index())
report["specparam_good_fit_share"] = report["specparam_good_fit_share"].round(2)
report["specparam_status"] = report["specparam_good_fit_share"].map(
    lambda r: "Keep" if r >= _g["min_good_channel_fraction"] else "Exclude")

# Fit error beside the R^2 rule, not part of it. R^2 rises with the exponent
# (a steeper spectrum leaves more variance to explain), the error does not, so
# a subject can pass the R^2 rule with poor fits (runs/v0.3.0-dev: sub-109).
# error_mae is specparam's mean absolute error of the fit (log10 power). Median
# over measured channels only (interpolated ones are averages of their
# neighbours). _z: robust z-score against the other subjects of this run
# (median and MAD); high means fits worse than the group.
err = per_channel[~per_channel["interpolated"]].groupby("subject_id")["error_mae"].median()
err_z = (err - err.median()) / (1.4826 * (err - err.median()).abs().median())
report["specparam_error_median"] = report["subject_id"].map(err).round(4)
report["specparam_error_z"] = report["subject_id"].map(err_z).round(1)
report = pd.concat([report, pd.DataFrame({"subject_id": missing,
                                          "specparam_status": "No results"})],
                   ignore_index=True)

# Counts from step 04 (bad channels, ICA components, autoreject), for exclusion
# decisions and as covariates.
report = report.merge(meta, on="subject_id", how="left")

print(report["specparam_status"].value_counts().to_string())
display(report.sort_values("specparam_good_fit_share"))

## Save

In [ ]:
all_rows.to_csv(out_dir / "all_subjects_specparam.csv", index=False)
per_channel.to_csv(out_dir / "unique_specparam_results.csv", index=False)
report.to_csv(out_dir / "participant_exclusion_report.csv", index=False)
for f in sorted(out_dir.glob("*.csv")):
    print(f)